# Deformation Descriptors

These descriptors compare a *deformed* configuration to a *reference*
(undeformed) configuration, quantifying local strain and non-affine
displacement:

| Descriptor | Returns | Reference |
|------------|---------|-----------|
| `atomic_strain` | 3x3 Green-Lagrange tensor | Shimizu et al. 2007 |
| `von_mises_strain` | scalar shear invariant | — |
| `d2min` | non-affine mean-squared displacement | Falk & Langer 1998 |
| `slip_vector` | 3D slip direction | Zimmerman et al. 2001 |

In [1]:
import numpy as np
import pyscal3 as pc
from ase.build import bulk

## 1. Uniaxial tension

In [2]:
# Reference (undeformed) FCC Al
ref = bulk("Al", cubic=True).repeat(3)
pc.find_neighbors(ref, method="cutoff", cutoff=4.0)

# 5% uniaxial strain in x
deformed = ref.copy()
cell = deformed.get_cell()
cell[0, 0] *= 1.05
deformed.set_cell(cell, scale_atoms=True)
pc.find_neighbors(deformed, method="cutoff", cutoff=4.0)

E = pc.atomic_strain(deformed, ref)
print("Mean strain tensor:")
print(np.round(np.nanmean(E, axis=0), 4))

Mean strain tensor:
[[ 0.0512 -0.     -0.    ]
 [-0.     -0.     -0.    ]
 [-0.     -0.     -0.    ]]


In [3]:
vm = pc.von_mises_strain(deformed, ref)
print("Mean von Mises strain: {:.4f}".format(np.nanmean(vm)))

Mean von Mises strain: 0.0512


## 2. D^2_min: Non-affine displacement

In [4]:
# Affine deformation: D^2_min ~ 0
d2_affine = pc.d2min(deformed, ref)
print("Affine deformation D^2_min: {:.6f}".format(np.nanmean(d2_affine)))

Affine deformation D^2_min: 0.000000


In [5]:
# Add random non-affine component
noisy = deformed.copy()
rng = np.random.default_rng(42)
noisy.positions += rng.normal(0, 0.1, noisy.positions.shape)
pc.find_neighbors(noisy, method="cutoff", cutoff=4.0)

d2_noisy = pc.d2min(noisy, ref)
print("With noise D^2_min: {:.4f}".format(np.nanmean(d2_noisy)))

With noise D^2_min: 0.0000


## 3. Slip vector

In [6]:
sv = pc.slip_vector(deformed, ref)
print("Mean slip vector:", np.round(np.nanmean(sv, axis=0), 4))
print("Slip vector magnitude:", np.round(np.nanmean(np.linalg.norm(sv, axis=1)), 4))

Mean slip vector: [0. 0. 0.]
Slip vector magnitude: 0.0


## 4. Simple shear

Shear deformation should give non-zero von Mises strain even for
isochoric (volume-preserving) deformation.

In [7]:
sheared = ref.copy()
pos = sheared.get_positions()
pos[:, 0] += 0.03 * pos[:, 1]  # 3% shear in xy
sheared.set_positions(pos)
pc.find_neighbors(sheared, method="cutoff", cutoff=4.0)

E_shear = pc.atomic_strain(sheared, ref)
vm_shear = pc.von_mises_strain(sheared, ref)
print("Shear Exy: {:.4f}".format(np.nanmean(E_shear[:, 0, 1])))
print("von Mises: {:.4f}".format(np.nanmean(vm_shear)))

Shear Exy: 0.0000
von Mises: 0.0200


## Notes

* **Reference configuration must have the same neighbor list** (use the
  same cutoff and method for both).
* **At least 3 common neighbors** are required to fit the deformation
  gradient; atoms with fewer neighbors return NaN.
* These descriptors are pure Python (small matrix operations per atom)
  and may be slow for very large systems.